# Mon Compagnon Immobilier — Pipeline territorial de bout en bout (millésime déclaré 2026)

**Objet :** repartir des fichiers bruts conservés dans `data/raw/`, appliquer les étapes de normalisation et d’enrichissement, contrôler les résultats puis exporter le référentiel géographique unifié.

## Périmètre réellement couvert
1. Lecture et contrôle des sources brutes : Code officiel géographique (COG), composition communale des bassins de vie 2022 en géographie au 01/01/2026, archive des populations légales.
2. Construction du référentiel communal `COG + bassin de vie`.
3. Enrichissement par les populations `PMUN`, `PCAP`, `PTOT` avec diagnostic des écarts.
4. Construction des 45 arrondissements municipaux à partir du COG et rattachement au parent communal / bassin de vie.
5. Unification, contrôles bloquants, rapports, manifeste des sources, empreintes SHA-256 et exports.

**Important :** le millésime 2026 désigne ici la géographie / édition des fichiers retenus, pas nécessairement l’année statistique de référence de chaque indicateur. Les sources complémentaires BPE, Filosofi, mobilités et DVF ne sont pas intégrées dans ce pipeline ; elles doivent être traitées comme des lots séparés après validation de leurs fichiers et dictionnaires. Le fichier `communes-france-2026.csv` n’est pas utilisé car ce n’est pas la source de référence de cette construction.


## 0. Configuration et chemins
Le notebook est autonome dans le dossier de livraison. Il ne modifie jamais les fichiers bruts.

In [6]:

from pathlib import Path
import sys
import pandas as pd

# Recherche automatique de la racine du projet
CANDIDATES = [
    Path.cwd(),
    Path("/content"),
    Path("/content/Mon_Compagnon_Immobilier_E2E_2026"),
    Path("/content/Mon_Compagnon_Immobilier_Pipeline_E2E_2026"),
]

BASE_DIR = next(
    (
        p for p in CANDIDATES
        if (p / "data" / "raw" / "v_commune_2026.csv").exists()
    ),
    None,
)

# Si les fichiers bruts sont directement à la racine
if BASE_DIR is None:
    BASE_DIR = next(
        (
            p for p in CANDIDATES
            if (p / "v_commune_2026.csv").exists()
        ),
        None,
    )

if BASE_DIR is None:
    raise FileNotFoundError(
        "Impossible de trouver v_commune_2026.csv. "
        "Vérifie le dossier contenant les fichiers bruts."
    )

# Détection de l'emplacement des sources
if (BASE_DIR / "data" / "raw").exists():
    RAW_DIR = BASE_DIR / "data" / "raw"
else:
    RAW_DIR = BASE_DIR

OUT_DIR = BASE_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

COG_FILE = RAW_DIR / "v_commune_2026.csv"
BV_FILE = RAW_DIR / "BV2022_au_01-01-2026.xlsx"
POP_ZIP = RAW_DIR / "ensemble.zip"

# Vérification des sources indispensables
for p in [COG_FILE, BV_FILE, POP_ZIP]:
    if not p.exists():
        raise FileNotFoundError(
            f"Source brute manquante : {p}"
        )

print("Racine du projet :", BASE_DIR.resolve())
print("Dossier des sources :", RAW_DIR.resolve())
print("Dossier des sorties :", OUT_DIR.resolve())
print("Python :", sys.version.split()[0])
print("Pandas :", pd.__version__)

print("\nSources détectées :")
for p in [COG_FILE, BV_FILE, POP_ZIP]:
    print(f"OK — {p.name} ({p.stat().st_size:,} octets)")


Racine du projet : /content
Dossier des sources : /content
Dossier des sorties : /content/outputs
Python : 3.13.18
Pandas : 2.2.3

Sources détectées :
OK — v_commune_2026.csv (3,565,935 octets)
OK — BV2022_au_01-01-2026.xlsx (1,399,909 octets)
OK — ensemble.zip (1,032,385 octets)


## 1. Inventaire, empreintes et inspection des sources brutes

In [7]:
def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""): h.update(chunk)
    return h.hexdigest()

inventory=[]
for p in sorted(RAW_DIR.iterdir()):
    if p.is_file(): inventory.append({"fichier":p.name,"taille_octets":p.stat().st_size,"sha256":sha256_file(p)})
manifest = pd.DataFrame(inventory)
display(manifest)
manifest.to_csv(OUT_DIR / "manifest_sources_brutes_sha256.csv", sep=";", index=False, encoding="utf-8-sig")

cog = pd.read_csv(COG_FILE, sep=None, engine="python", dtype="string", encoding="utf-8-sig", keep_default_na=False)
cog.columns = cog.columns.str.replace("\ufeff", "", regex=False).str.strip()
print("COG dimensions :", cog.shape, "Types :", cog["TYPECOM"].value_counts().to_dict())
with zipfile.ZipFile(POP_ZIP) as z:
    zip_listing = pd.DataFrame([{"fichier":i.filename,"taille_octets":i.file_size} for i in z.infolist()])
    print("Contenu de l'archive population :")
    display(zip_listing)
    if "donnees_communes.csv" not in z.namelist(): raise ValueError("donnees_communes.csv absent de l'archive")
    pop = pd.read_csv(z.open("donnees_communes.csv"), sep=";", dtype="string", encoding="utf-8-sig", keep_default_na=False)


,fichier,taille_octets,sha256
0,BV2022_au_01-01-2026.xlsx,1399909,28f1e9b78ff42ab29d8cc75a71dc59bd356cb0f22d00f5...
1,ensemble.zip,1032385,5174fc225398a4c072fb8bb5298b7ccf30c017834c7a41...
2,v_commune_2026.csv,3565935,7be1153de3a63df48fa0ca77f8490c8a0c5c9c4eb12f5c...


COG dimensions : (37496, 12) Types : {'COM': 34875, 'COMD': 2105, 'COMA': 471, 'ARM': 45}
Contenu de l'archive population :


,fichier,taille_octets
0,donnees_arrondissements.csv,22358
1,donnees_cantons.csv,120809
2,donnees_collectivites.csv,312
3,donnees_communes.csv,2167955
4,donnees_communes_deleguees.csv,179373
5,donnees_departements.csv,5753
6,donnees_fractions_cantonales.csv,53635
7,donnees_regions.csv,801
8,metadonnees.csv,1548324


## 2. Construire la table de composition communale des bassins de vie

In [8]:
# La feuille comporte des lignes de titre avant la ligne d'en-tête : la structure est contrôlée avant extraction.
bv_raw = pd.read_excel(BV_FILE, sheet_name="Composition_communale", header=None, dtype="string")
header_idx = None
for i in range(min(15, len(bv_raw))):
    vals = set(bv_raw.iloc[i].fillna("").astype(str).str.strip().tolist())
    if {"CODGEO", "BV2022", "LIBBV2022"}.issubset(vals):
        header_idx = i; break
if header_idx is None: raise ValueError("En-tête CODGEO/BV2022/LIBBV2022 introuvable dans la feuille Composition_communale")
bv = pd.read_excel(BV_FILE, sheet_name="Composition_communale", header=header_idx, dtype="string")
bv.columns = bv.columns.astype(str).str.strip()
rename = {
    "Code géographique":"CODGEO", "Libellé géographique":"LIBELLE_BV_SOURCE",
    "Bassin de vie 2022":"BV2022", "Libellé géographique du bassin de vie 2022":"LIBBV2022",
    "Catégorie de la commune":"TYPE_COMMUNE_BV2022", "Département":"DEP_BV", "Région":"REG_BV"
}
bv = bv.rename(columns=rename)
required_bv={"CODGEO","BV2022","LIBBV2022"}
if not required_bv.issubset(bv.columns): raise ValueError(f"Colonnes BV absentes : {required_bv-set(bv.columns)}; colonnes lues : {bv.columns.tolist()}")
for c in ["CODGEO","BV2022","LIBBV2022"]: bv[c]=bv[c].astype("string").str.strip()
bv = bv[bv["CODGEO"].notna() & bv["CODGEO"].ne("")].copy()
if bv["CODGEO"].duplicated().any(): raise ValueError("CODGEO dupliqué dans la composition communale des bassins de vie")
print("Composition BV :", bv.shape, "codes uniques :", bv.CODGEO.nunique())
display(bv.head())


Composition BV : (34875, 7) codes uniques : 34875


,CODGEO,LIBGEO,BV2022,LIBBV2022,TYPE_COMMUNE_BV2022,DEP,REG
0,01001,L'Abergement-Clémenciat,01093,Châtillon-sur-Chalaronne,00,01,84
1,01002,L'Abergement-de-Varey,01004,Ambérieu-en-Bugey,00,01,84
2,01004,Ambérieu-en-Bugey,01004,Ambérieu-en-Bugey,20,01,84
3,01005,Ambérieux-en-Dombes,01333,Saint-André-de-Corcy,00,01,84
4,01006,Ambléon,01034,Belley,00,01,84


## 3. Construire le référentiel communal COG + bassin de vie

In [9]:
communes_cog = cog.loc[cog["TYPECOM"].eq("COM")].copy()
communes_cog = communes_cog.rename(columns={"COM":"CODGEO", "LIBELLE":"LIBELLE_COG"})
if communes_cog["CODGEO"].eq("").any() or communes_cog["CODGEO"].duplicated().any():
    raise ValueError("Clé CODGEO vide ou dupliquée dans les communes COG")
# Jointure gauche : on conserve le périmètre du COG, même si un bassin manque.
communes = communes_cog.merge(
    bv[[c for c in ["CODGEO","BV2022","LIBBV2022","TYPE_COMMUNE_BV2022"] if c in bv.columns]],
    on="CODGEO", how="left", validate="one_to_one", indicator="_statut_bv"
)
communes["STATUT_RAPPROCHEMENT_BV"] = np.where(communes["_statut_bv"].eq("both"), "APPARIE", "NON_APPARIE")
communes = communes.drop(columns="_statut_bv")
print("Communes COG :", len(communes), "; bassins appariés :", int(communes.STATUT_RAPPROCHEMENT_BV.eq("APPARIE").sum()))
display(communes["STATUT_RAPPROCHEMENT_BV"].value_counts(dropna=False).rename_axis("statut").to_frame("lignes"))


Communes COG : 34875 ; bassins appariés : 34875


,lignes
statut,
APPARIE,34875


## 4. Enrichir avec les populations légales et documenter les écarts

In [10]:
required_pop={"COM","Commune","PMUN","PCAP","PTOT"}
if not required_pop.issubset(pop.columns): raise ValueError(f"Colonnes population absentes : {required_pop-set(pop.columns)}")
pop = pop.copy()
pop["COM"] = pop["COM"].astype("string").str.strip()
if pop["COM"].eq("").any() or pop["COM"].duplicated().any():
    raise ValueError("COM vide ou dupliqué dans la source population; vérifier l'unité géographique avant fusion")
pop_join = pop[["COM","PMUN","PCAP","PTOT"]].rename(columns={"COM":"CODGEO_POP","PMUN":"POP_MUNICIPALE","PCAP":"POP_COMPTEE_A_PART","PTOT":"POP_TOTALE"})
communes_pop = communes.merge(pop_join, left_on="CODGEO", right_on="CODGEO_POP", how="left", validate="one_to_one", indicator="_statut_pop")
communes_pop["STATUT_RAPPROCHEMENT_POP"] = np.select(
    [communes_pop["_statut_pop"].eq("both"), communes_pop["CODGEO"].isin(["13055","69123","75056"])],
    ["APPARIE", "ECART_PERIMETRE_COMMUNE_AGREGEE"], default="NON_APPARIE"
)
communes_pop = communes_pop.drop(columns=["CODGEO_POP","_statut_pop"])
# Diagnostics des codes de part et d'autre; les écarts sont exportés, pas masqués.
ref_codes=set(communes["CODGEO"]); pop_codes=set(pop["COM"])
only_ref=sorted(ref_codes-pop_codes); only_pop=sorted(pop_codes-ref_codes)
ecarts=[]
for code_ in only_ref: ecarts.append({"code":code_,"cote":"COG_BV","diagnostic":"Absent de la source population"})
for code_ in only_pop: ecarts.append({"code":code_,"cote":"POPULATION","diagnostic":"Absent du référentiel des communes COG"})
ecarts_df=pd.DataFrame(ecarts, columns=["code","cote","diagnostic"])
ecarts_df.to_csv(OUT_DIR/"ecarts_perimetre_population.csv",sep=";",index=False,encoding="utf-8-sig")
print("Codes communs COG/BV absents population :",len(only_ref),"; codes population absents COG/BV :",len(only_pop))
print("Statuts population :")
display(communes_pop["STATUT_RAPPROCHEMENT_POP"].value_counts(dropna=False).rename_axis("statut").to_frame("lignes"))


Codes communs COG/BV absents population : 20 ; codes population absents COG/BV : 45
Statuts population :


,lignes
statut,
APPARIE,34855
NON_APPARIE,17
ECART_PERIMETRE_COMMUNE_AGREGEE,3


## 5. Construire les 45 arrondissements municipaux à partir du COG

In [11]:
arm_cog = cog.loc[cog["TYPECOM"].eq("ARM")].copy()
arm_cog = arm_cog.rename(columns={"COM":"CODE_ARRONDISSEMENT", "LIBELLE":"LIBELLE_ARRONDISSEMENT", "COMPARENT":"CODE_COMMUNE_PARENT"})
if len(arm_cog)!=45: raise ValueError(f"Nombre d'arrondissements municipaux inattendu : {len(arm_cog)} (attendu 45)")
if arm_cog["CODE_ARRONDISSEMENT"].duplicated().any(): raise ValueError("Code arrondissement dupliqué")
parents=communes_pop[["CODGEO","LIBELLE_COG","BV2022","LIBBV2022","DEP","REG"]].rename(columns={"CODGEO":"CODE_COMMUNE_PARENT","LIBELLE_COG":"LIBELLE_COMMUNE_PARENT","DEP":"DEP_PARENT","REG":"REG_PARENT"})
arm = arm_cog.merge(parents, on="CODE_COMMUNE_PARENT", how="left", validate="many_to_one", indicator="_parent")
if not arm["_parent"].eq("both").all(): raise ValueError("Un ou plusieurs arrondissements n'ont pas de commune parente appariée")
arm = arm.drop(columns="_parent")
# Population d'arrondissement non renseignée si aucune source officielle à ce niveau n'est intégrée.
arm["POP_MUNICIPALE"] = pd.NA
arm["POP_COMPTEE_A_PART"] = pd.NA
arm["POP_TOTALE"] = pd.NA
arm["STATUT_RAPPROCHEMENT_POP"] = "NON_DISPONIBLE_AU_NIVEAU_ARM"
arm["STATUT_RAPPROCHEMENT_BV"] = "HERITE_COMMUNE_PARENTE"
print("Arrondissements :",len(arm),"; par ville :")
display(arm["CODE_COMMUNE_PARENT"].value_counts().rename_axis("code_parent").to_frame("nombre_ARM"))


Arrondissements : 45 ; par ville :


,nombre_ARM
code_parent,
75056,20
13055,16
69123,9


## 6. Harmoniser les schémas et assembler le référentiel unifié

In [12]:
# Les colonnes du COG et les colonnes spécifiques aux ARM sont conservées dans le schéma commun.
communes_unifiees = communes_pop.copy()
communes_unifiees["ID_GEO"] = "COM_" + communes_unifiees["CODGEO"].astype(str)
communes_unifiees["NIVEAU_GEO"] = "COMMUNE"
communes_unifiees["CODE_COMMUNE_PARENT"] = pd.NA

# Préparer les lignes ARM avec toutes les colonnes du schéma communal; ne pas recopier la population du parent.
arm_unifie = pd.DataFrame(index=range(len(arm)), columns=communes_unifiees.columns)
for _, r in arm.iterrows():
    idx = r.name
    arm_unifie.loc[idx, "TYPECOM"] = "ARM"
    arm_unifie.loc[idx, "CODGEO"] = r["CODE_ARRONDISSEMENT"]
    arm_unifie.loc[idx, "LIBELLE_COG"] = r["LIBELLE_ARRONDISSEMENT"]
    arm_unifie.loc[idx, "DEP"] = r["DEP"]
    arm_unifie.loc[idx, "REG"] = r["REG"]
    arm_unifie.loc[idx, "COMPARENT"] = r["CODE_COMMUNE_PARENT"]
    arm_unifie.loc[idx, "BV2022"] = r["BV2022"]
    arm_unifie.loc[idx, "LIBBV2022"] = r["LIBBV2022"]
    arm_unifie.loc[idx, "STATUT_RAPPROCHEMENT_BV"] = "HERITE_COMMUNE_PARENTE"
    arm_unifie.loc[idx, "POP_MUNICIPALE"] = pd.NA
    arm_unifie.loc[idx, "POP_COMPTEE_A_PART"] = pd.NA
    arm_unifie.loc[idx, "POP_TOTALE"] = pd.NA
    arm_unifie.loc[idx, "STATUT_RAPPROCHEMENT_POP"] = "NON_DISPONIBLE_AU_NIVEAU_ARM"
    arm_unifie.loc[idx, "ID_GEO"] = "ARM_" + str(r["CODE_ARRONDISSEMENT"])
    arm_unifie.loc[idx, "NIVEAU_GEO"] = "ARRONDISSEMENT_MUNICIPAL"
    arm_unifie.loc[idx, "CODE_COMMUNE_PARENT"] = r["CODE_COMMUNE_PARENT"]

referentiel_unifie = pd.concat([communes_unifiees, arm_unifie], ignore_index=True, sort=False)
if referentiel_unifie["ID_GEO"].duplicated().any(): raise ValueError("ID_GEO non unique dans le référentiel unifié")
print("Dimensions référentiel unifié :", referentiel_unifie.shape)
display(referentiel_unifie["NIVEAU_GEO"].value_counts().rename_axis("niveau").to_frame("lignes"))


Dimensions référentiel unifié : (34920, 23)


/tmp/ipykernel_636/3205698492.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  referentiel_unifie = pd.concat([communes_unifiees, arm_unifie], ignore_index=True, sort=False)


,lignes
niveau,
COMMUNE,34875
ARRONDISSEMENT_MUNICIPAL,45


## 7. Contrôles qualité bloquants et rapports

In [13]:
def ctl(nom, ok, detail): return {"CONTROLE":nom,"RESULTAT":"OK" if bool(ok) else "ECHEC","DETAIL":str(detail)}
arm_rows=referentiel_unifie[referentiel_unifie["NIVEAU_GEO"].eq("ARRONDISSEMENT_MUNICIPAL")]
com_rows=referentiel_unifie[referentiel_unifie["NIVEAU_GEO"].eq("COMMUNE")]
controls=[
 ctl("COG communes attendu 34875",len(communes)==34875,f"observé={len(communes)}"),
 ctl("Composition BV clé unique",bv["CODGEO"].is_unique,f"lignes={len(bv)}"),
 ctl("Communes clé unique",communes_pop["CODGEO"].is_unique,f"lignes={len(communes_pop)}"),
 ctl("Toutes les communes conservées",len(com_rows)==len(communes_pop),f"communes={len(com_rows)} source={len(communes_pop)}"),
 ctl("45 arrondissements municipaux",len(arm_rows)==45,f"observé={len(arm_rows)}"),
 ctl("ID_GEO unique",referentiel_unifie["ID_GEO"].is_unique,f"lignes={len(referentiel_unifie)}"),
 ctl("Tous les ARM ont un parent",arm_rows["CODE_COMMUNE_PARENT"].notna().all(),f"parents renseignés={arm_rows['CODE_COMMUNE_PARENT'].notna().sum()}"),
 ctl("Population ARM non imputée",arm_rows[["POP_MUNICIPALE","POP_COMPTEE_A_PART","POP_TOTALE"]].isna().all().all(),"aucune population parent copiée"),
 ctl("Bassin présent pour tous les ARM",arm_rows["BV2022"].notna().all(),f"BV renseignés={arm_rows['BV2022'].notna().sum()}"),
 ctl("Statuts population explicites",com_rows["STATUT_RAPPROCHEMENT_POP"].notna().all(),"statut non nul sur toutes les communes"),
 ctl("Clés territoriales texte",referentiel_unifie["CODGEO"].map(lambda x:isinstance(x,str)).all(),"CODGEO lu comme chaîne")
]
rapport = pd.DataFrame(controls)
display(rapport)
rapport.to_csv(OUT_DIR/"rapport_controles_pipeline_e2e.csv",sep=";",index=False,encoding="utf-8-sig")
if not rapport["RESULTAT"].eq("OK").all(): raise AssertionError("Contrôle bloquant en échec : export final interdit")


,CONTROLE,RESULTAT,DETAIL
0,COG communes attendu 34875,OK,observé=34875
1,Composition BV clé unique,OK,lignes=34875
2,Communes clé unique,OK,lignes=34875
3,Toutes les communes conservées,OK,communes=34875 source=34875
4,45 arrondissements municipaux,OK,observé=45
5,ID_GEO unique,OK,lignes=34920
6,Tous les ARM ont un parent,OK,parents renseignés=45
7,Population ARM non imputée,OK,aucune population parent copiée
8,Bassin présent pour tous les ARM,OK,BV renseignés=45
9,Statuts population explicites,OK,statut non nul sur toutes les communes


## 8. Export des tables intermédiaires et finales

In [14]:
def export_csv(df, name):
    path=OUT_DIR/name
    df.to_csv(path,sep=";",index=False,encoding="utf-8-sig")
    # Relecture indépendante et contrôle des dimensions.
    check=pd.read_csv(path,sep=";",dtype="string",encoding="utf-8-sig",keep_default_na=False)
    if check.shape != df.shape: raise AssertionError(f"Relecture différente pour {name}: {check.shape} != {df.shape}")
    return path

exports = [
    export_csv(communes, "etape_1_communes_cog_bassins_vie.csv"),
    export_csv(communes_pop, "etape_2_communes_bassins_vie_population.csv"),
    export_csv(arm, "etape_3_arrondissements_municipaux.csv"),
    export_csv(referentiel_unifie, "referentiel_geographique_unifie_2026.csv"),
]
# Rapport final d'audit compact.
audit = pd.DataFrame([
 {"indicateur":"lignes_communes","valeur":len(communes_pop)},
 {"indicateur":"lignes_arrondissements_municipaux","valeur":len(arm)},
 {"indicateur":"lignes_referentiel_unifie","valeur":len(referentiel_unifie)},
 {"indicateur":"communes_population_appariee","valeur":int(communes_pop["STATUT_RAPPROCHEMENT_POP"].eq("APPARIE").sum())},
 {"indicateur":"communes_population_non_appariee","valeur":int(communes_pop["STATUT_RAPPROCHEMENT_POP"].eq("NON_APPARIE").sum())},
 {"indicateur":"communes_population_ecart_perimetre","valeur":int(communes_pop["STATUT_RAPPROCHEMENT_POP"].eq("ECART_PERIMETRE_COMMUNE_AGREGEE").sum())},
 {"indicateur":"controles_OK","valeur":int(rapport["RESULTAT"].eq("OK").sum())},
 {"indicateur":"controles_ECHEC","valeur":int(rapport["RESULTAT"].eq("ECHEC").sum())},
])
export_csv(audit,"audit_synthese_pipeline_e2e.csv")
# Empreinte de la sortie finale.
final_path=OUT_DIR/"referentiel_geographique_unifie_2026.csv"
(OUT_DIR/"SHA256_referentiel_geographique_unifie_2026.txt").write_text(f"SHA256  {sha256_file(final_path)}  {final_path.name}\n",encoding="utf-8")
print("Exports générés :")
for p in exports: print(" -",p.name,p.stat().st_size,"octets")
print("Pipeline terminé. Statut technique :", "OK" if rapport.RESULTAT.eq("OK").all() else "ECHEC")


Exports générés :
 - etape_1_communes_cog_bassins_vie.csv 3456424 octets
 - etape_2_communes_bassins_vie_population.csv 4119059 octets
 - etape_3_arrondissements_municipaux.csv 9065 octets
 - referentiel_geographique_unifie_2026.csv 4789098 octets
Pipeline terminé. Statut technique : OK


In [15]:

from pathlib import Path

print("Dossier de travail actuel :", Path.cwd())
print("\nFichiers exportés retrouvés :")

noms = [
    "etape_1_communes_cog_bassins_vie.csv",
    "etape_2_communes_bassins_vie_population.csv",
    "etape_3_arrondissements_municipaux.csv",
    "referentiel_geographique_unifie_2026.csv",
]

racines = [
    Path.cwd(),
    Path("/content"),
]

trouves = set()

for racine in racines:
    if racine.exists():
        for fichier in racine.rglob("*"):
            if fichier.is_file() and fichier.name in noms:
                trouves.add(fichier.resolve())

if trouves:
    for fichier in sorted(trouves):
        print(f"OK — {fichier}")
else:
    print("Aucun des quatre fichiers retrouvé dans les emplacements explorés.")

Dossier de travail actuel : /content

Fichiers exportés retrouvés :
OK — /content/outputs/etape_1_communes_cog_bassins_vie.csv
OK — /content/outputs/etape_2_communes_bassins_vie_population.csv
OK — /content/outputs/etape_3_arrondissements_municipaux.csv
OK — /content/outputs/referentiel_geographique_unifie_2026.csv


In [16]:

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

fichiers = [
    p for p in trouves
    if p.name in noms
]

if len({p.name for p in fichiers}) != 4:
    raise RuntimeError(
        "Les quatre exports ne sont pas tous disponibles. "
        "Vérifie leur emplacement avant de créer l'archive."
    )

zip_path = Path("/content/exports_referentiel_2026.zip")

with ZipFile(zip_path, "w", ZIP_DEFLATED) as archive:
    for fichier in fichiers:
        archive.write(fichier, arcname=fichier.name)

print("Archive créée :", zip_path)

Archive créée : /content/exports_referentiel_2026.zip


## 9. Registre des sources et limites de traçabilité

| Source brute | Rôle | Millésime / géographie | Limite à conserver |
|---|---|---|---|
| `v_commune_2026.csv` | COG : codes, types, libellés, parents | Édition 2026 | Conserver la documentation officielle et la date de téléchargement dans le registre du projet |
| `BV2022_au_01-01-2026.xlsx` | Composition communale des bassins de vie | Bassins de vie 2022, géographie au 01/01/2026 | Année du bassin et date de géographie ne sont pas la même chose |
| `ensemble.zip` | Population communale : PMUN, PCAP, PTOT | Archive population fournie | Le millésime de diffusion n’implique pas une année statistique 2026 |

Les chemins, tailles et SHA-256 des sources sont consignés dans `manifest_sources_brutes_sha256.csv`. Pour une traçabilité officielle complète, compléter dans un registre séparé les URL de téléchargement exactes, dates de récupération et références documentaires officielles. Les valeurs ne sont pas recalculées ni imputées. Les 3 communes agrégées de Paris, Lyon et Marseille font l’objet d’un statut d’écart de périmètre si leur code ne correspond pas à la table population.

### Ce que ce pipeline ne fait pas encore
- Il ne construit pas les variables socio-économiques BPE / Filosofi / mobilités : les fichiers et schémas validés ne sont pas présents dans le lot brut actuel.
- Il ne charge pas les transactions DVF et ne calcule aucun prix immobilier.
- Il ne produit pas de cible de machine learning ; la référence territoriale est une couche de jointure, pas une estimation de prix.
